# Extended benchmark algorithm comparison

This notebook compares the scalable methods **Greedy, Heuristic, Uniform, and Random** on one-dimensional and two-dimensional datasets generated from unfiltered random field draws. Exhaustive is excluded because enumerating all allocations and adaptive trees is intended only for the compact benchmark.

Greedy allocates each sensor where it gives the largest average pairwise error reduction at a location and then builds a local minimum-$P_{ijmn}$ tree. Uniform uses a near-uniform allocation and a seeded random pair-location tree. Random uses both a seeded random allocation and tree. Every completed tree is evaluated with the full $P/Q$ branch model under equal field priors.

In [ ]:
# Resolve the project whether Jupyter starts here or from the repository root.
import importlib
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

working = Path.cwd().resolve()
PROJECT = working / "problem-2-min-error-seq" if (working / "problem-2-min-error-seq").is_dir() else working
SOURCE = PROJECT / "src"
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))

_analysis_module = importlib.reload(importlib.import_module("analysis"))
run_analysis = _analysis_module.run_analysis
_figures_module = importlib.reload(importlib.import_module("figures"))
save_figures = _figures_module.save_figures
summary_markdown = _figures_module.summary_markdown

## Experimental settings

`test` mode selects **180 cases per spatial dimension** and covers every $(K,M)$ pair for $K\in\{3,\ldots,10\}$ and $M\in\{2,\ldots,10\}$. Each pair contributes two or three cases, and each field family contributes 60 cases. `full` mode uses all 6,480 cases in each dimension.

The time limit applies independently to each method-budget evaluation, and checkpoint reuse supports interrupted runs. Progress is printed after every completed case, separately for each dataset.

In [ ]:
# Use "full" for the complete study after confirming the test run.
RUN_MODE = "test"
RESUME = True
TIME_LIMIT_SECONDS = 30.0

# These controls apply only to the heuristic method.
BEAM_WIDTH = 3
MAX_ITERATIONS = 30

# This notebook is intentionally tied to the extended datasets.
STUDY = "extended"

## Run the matched comparison

Greedy, Heuristic, Uniform, and Random are evaluated on identical selected case-budget combinations. Results are checkpointed after every method-budget evaluation, and aggregation uses only combinations completed by all four methods.

In [ ]:
# Run or resume the extended comparison with the settings above.
import importlib
run_analysis = importlib.reload(importlib.import_module("analysis")).run_analysis
analysis = run_analysis(
    project=PROJECT,
    study=STUDY,
    mode=RUN_MODE,
    resume=RESUME,
    exact_k_max=4,  # Accepted by the shared runner; Exhaustive is not used here.
    time_limit=TIME_LIMIT_SECONDS,
    beam_width=BEAM_WIDTH,
    max_iterations=MAX_ITERATIONS,
)

# Confirm that this notebook used both extended datasets and the four scalable methods.
expected_datasets = {"benchmark_1d", "benchmark_2d"}
expected_methods = {"Greedy", "Heuristic", "Uniform", "Random"}
observed_datasets = {row["dataset"] for row in analysis["rows"]}
observed_methods = {row["method"] for row in analysis["rows"]}
assert observed_datasets == expected_datasets
assert observed_methods == expected_methods
expected_k = set(range(3, 11))
expected_m = set(range(2, 11))
assert {int(row["K"]) for row in analysis["rows"]} == expected_k
assert {int(row["M"]) for row in analysis["rows"]} == expected_m
assert {(int(row["K"]), int(row["M"])) for row in analysis["rows"]} == {(K, M) for K in expected_k for M in expected_m}

case_counts = {
    dimension: len({
        (row["dataset"], row["case"])
        for row in analysis["rows"]
        if int(row["dimension"]) == dimension
    })
    for dimension in (1, 2)
}
if RUN_MODE == "test":
    assert min(case_counts.values()) >= 180

print("Verified datasets:", sorted(observed_datasets))
print("Verified methods:", sorted(observed_methods))
print("Unique cases by dimension:", case_counts)
print("Results directory:", analysis["output"])

## Completion and matched summary

The completion counts are shown before aggregation. The summary table reports matched sample sizes, mean budget fraction, mean error probability, and mean runtime.

In [ ]:
# Report method completion before displaying aggregate values.
from collections import Counter

completion = Counter((row["method"], row["status"]) for row in analysis["rows"])
print("Method-status counts:", completion)
display(Markdown(summary_markdown(analysis["summary"])))

## Comparison figures

The figures show **mean error probability versus mean budget fraction**, runtime distributions by $K$, and error distributions by field family. Runtime and field-family distributions use grouped box plots. All figures use the matched samples summarized above.

In [ ]:
# Save publication-quality PDF and PNG versions, then display each figure.
save_figures = importlib.reload(importlib.import_module("figures")).save_figures
if not analysis["summary"]:
    raise RuntimeError("No matched completed cases are available for plotting")

figures = save_figures(
    analysis["summary"],
    analysis["rows"],
    analysis["output"],
)
for name, figure in figures.items():
    display(Markdown(f"### {name.replace('_', ' ').title()}"))
    display(figure)
    plt.close(figure)